# 1. Загрузка данных

In [ ]:
# Библиотеки для работы с данными
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Библиотеки для работы с картами
import folium
import geohash2
from folium import Element
import geopandas as gpd

import ast
from scipy import stats
from scipy.stats import spearmanr
from scipy.stats import kruskal
import statsmodels.formula.api as smf
from libpysal.weights import KNN
from esda.moran import Moran_Local
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
import json
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
from libpysal.weights import KNN
from esda.moran import Moran

In [ ]:
#Загрузим даатсет с вакансиями
data = pd.read_csv ('external_data\hh_2024_sample.csv', low_memory=False)

In [ ]:
data = data.drop(['premium',
              'relations',
              'insider_interview',
              'response_letter_required',
              'salary',
              'allow_messages',
              'department',
              'contacts',
              'vacancy_constructor_template',
              'archived',
              'specializations',
              'code',
              'hidden',
              'quick_responses_allowed',
              'accept_incomplete_resumes',
              'created_at',
              'initial_created_at',
              'negotiations_url',
              'suitable_resumes_url',
              'apply_alternate_url',
              'test',
              'address.description',
              'address.metro',
              'employer.logo_urls',
              'department.id',
              'year'], axis=1)

In [ ]:
data = data.drop_duplicates(subset='id', keep='first')

In [ ]:
duplicate_description = data.duplicated(subset=['name', 
                                       'description', 
                                       'branded_description',
                                       'key_skills',
                                       'professional_roles',
                                       'area.id',
                                       'address.city',
                                       'experience.id',
                                       'employment.id',
                                       'employer.id',
                                       'salary.from',
                                       'salary.gross',
                                        ### вторичный уровень влияния
                                       'accept_handicapped',
                                       'accept_kids',
                                       'working_days',
                                       'working_time_intervals',
                                       'working_time_modes',
                                       'accept_temporary',
                                       'languages',
                                       'driver_license_types',
                                       'schedule.id',
                                       'internship',
                                       'night_shifts',
                                       'work_format',
                                       'working_hours' ], keep='first')

In [ ]:
data = data[~duplicate_description]

In [ ]:
#Подгрузим данные о регионах РФ, чтобы оставить в выборке только те вакансии, которые относятся к РФ
data_prof = pd.read_excel('external_data\data_RUR_with_region_2704.xlsx')

In [ ]:
df = data.merge(data_prof, on='id', how='inner')

In [ ]:
df = df[df['id'] != 99159786]

In [ ]:
if isinstance(df['professional_roles'].iloc[0], str):
    df['professional_roles'] = df['professional_roles'].apply(ast.literal_eval)

In [ ]:
df['role_id'] = df['professional_roles'].apply(lambda x: x[0]['id'])
df['role_name'] = df['professional_roles'].apply(lambda x: x[0]['name'])

In [ ]:
#выделим роли, которые относятся к отрасли Торговля
target_ids = {'97','70','9','35','129','106'}
df_prod = df[df['role_id'].isin(target_ids)].copy()

In [ ]:
#Создадим итоговый датасет 
df_prod = df_prod[['name',
              'description',
              'role_id',
              'role_name',
              'key_skills',
              'experience.id',
              'experience.name',
              'schedule.id',
              'schedule.name',
              'employment.id',
              'employment.name',
              'address.lat_2',
              'address.lng_2',
              'address.city_new',
              'region_name',
              'salary.from',
              'salary.gross']].copy()

# 2. Анализ данных

In [ ]:
df_prod.info()

In [ ]:
df_prod = df_prod.rename(columns={
    'experience.name': 'experience_name',
    'experience.id': 'experience_id',
    'schedule.id': 'schedule_id',
    'employment.id': 'employment_id'})

In [ ]:
df_prod.duplicated().sum()

In [ ]:
#удалим полные дубликаты
df_prod = df_prod.drop_duplicates().copy()

In [ ]:
df_prod.shape

In [ ]:
#Проверим пропуски
sns.heatmap(df_prod.isna().T)
plt.title('Тепловая карта пропусков значений')
plt.show()

Пропуски есть в признаках:
* key_skills - с информацией о навыках
* address.lat_2, address.lng_2, address.city_new - точных координатах вакансии и названии населенного пункта

In [ ]:
#Проверим % пропусков для 'address.lat_2', 'address.lng_2', 'address.city_new'
cols_to_check = ['address.lat_2', 'address.lng_2', 'address.city_new']

for col in cols_to_check:
    print(col)
    print('Пропусков:', df_prod[col].isna().sum())
    print('Доля:', round(df_prod[col].isna().mean() * 100, 2), '%')
    print('---')

In [ ]:
#Проверим что пропуски по координатам почти полностью совпадают с пропуками по имени населенного пункта
(df_prod['address.lat_2'].isna() != df_prod['address.city_new'].isna()).sum()

In [ ]:
cols_to_check = ['role_id', 'role_name', 'experience_id', 'experience_name',
                 'schedule_id', 'schedule.name', 'employment_id', 'employment.name']

In [ ]:
for col in cols_to_check:
    unique_vals = df_prod[col].unique()
    num_unique = df_prod[col].nunique()
    print(f"Признак: {col}")
    print(f"Количество уникальных значений: {num_unique}")
    print(f"Уникальные значения: {unique_vals}")
    print("-" * 50)

In [ ]:
#закодируем опыт
experience_mapping = {
    'noExperience': 0,
    'between1And3': 1,
    'between3And6': 2,
    'moreThan6': 3
}
df_prod['experience_ord'] = df_prod['experience_id'].map(experience_mapping)

Проанализируем зарплату

In [ ]:
df_prod['salary.gross'].value_counts(dropna=False)

In [ ]:
#Удалим строки, c Nan
df_prod = df_prod[df_prod['salary.gross'].notna()].copy()

In [ ]:
df_prod.shape

In [ ]:
# Для встрех строк, где зарплата указана с учетом вычета налога - преобразуем в зарплату до вычета налога
TAX_RATE = 0.13

mask_net = df_prod['salary.gross'] == False

df_prod.loc[mask_net, 'salary_from_adj'] = round(df_prod.loc[mask_net, 'salary.from'] / (1 - TAX_RATE),0)
df_prod.loc[~mask_net, 'salary_from_adj'] = df_prod.loc[~mask_net, 'salary.from']

In [ ]:
#посмотрим количество строк с зарплатой меньше мрот 19 242
df_prod[df_prod['salary_from_adj'] < 19242].shape

In [ ]:
# посмотрим распределение зарплат меньше мрот 19 242
plt.figure(figsize=(8, 6))
sns.histplot(df_prod[df_prod['salary_from_adj'] < 19242]['salary_from_adj'], bins = 40) 
plt.title('распределение зарплат меньше мрот 19 242', fontsize=14)
plt.xlabel('Заработная плата', fontsize=14)
plt.ylabel('Количество вакансий', fontsize=14)

In [ ]:
# удалим строки с зарплатой меньше мрот 19 242
df_prod = df_prod[df_prod['salary_from_adj'] >= 19242].copy()

In [ ]:
df_prod.shape

In [ ]:
df_prod = df_prod[df_prod['salary_from_adj'] < 10000000]

In [ ]:
#Анализ по регионам топ-200 по количеству вакансий
top_regions = df_prod['region_name'].value_counts().head(200).index

plt.figure(figsize=(24,24))
sns.boxplot(
    data=df_prod[df_prod['region_name'].isin(top_regions)],
    x='salary_from_adj',
    y='region_name'
)
#plt.xlim(0, 6)
plt.show()

In [ ]:
#Анализ по регионам топ-200 по количеству вакансий
top_regions = df_prod['experience_name'].value_counts().head(200).index

plt.figure(figsize=(24,24))
sns.boxplot(
    data=df_prod[df_prod['experience_name'].isin(top_regions)],
    x='salary_from_adj',
    y='experience_name'
)
#plt.xlim(0, 6)
plt.show()

In [ ]:
#Анализ по регионам топ-200 по количеству вакансий
top_regions = df_prod['role_name'].value_counts().head(200).index

plt.figure(figsize=(24,24))
sns.boxplot(
    data=df_prod[df_prod['role_name'].isin(top_regions)],
    x='salary_from_adj',
    y='role_name'
)
#plt.xlim(0, 6)
plt.show()

In [ ]:
print("Исходное количество записей:", df_prod.shape[0])

In [ ]:
#удалим выбросы по зп в группах ('region_name', 'role_name', 'experience_name')
min_group_size = 10
def remove_outliers_iqr_safe(group, col='salary_from_adj'):
    if len(group) < min_group_size:
        return group
    q1 = group[col].quantile(0.25)
    q3 = group[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return group[(group[col] >= lower) & (group[col] <= upper)]

df_prod = df_prod.groupby(
    ['region_name', 'role_name', 'experience_name'], group_keys=False
).apply(remove_outliers_iqr_safe)

In [ ]:
#Фильтрация зарплаты `salary_from_adj`: для каждой группы с размером ≥ 10 
# oтавлены значения в интервале [Q1−1,5⋅IQR, Q3+1,5⋅IQR][Q1​−1,5⋅IQR,Q3​+1,5⋅IQR]; 
# малые группы без усечения
print("После удаления выбросов:", df_prod.shape[0])

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df_prod['salary_from_adj'], bins = 40) 
plt.title('Распределение зарплат', fontsize=14)
plt.xlabel('Заработная плата', fontsize=14)
plt.ylabel('Количество вакансий', fontsize=14)

In [ ]:
plt.figure(figsize=(24,24))
sns.boxplot(
    data=df_prod,
    x='salary_from_adj'
)

In [ ]:
#создалим новый столбец salary_from_log
df_prod['salary_from_log'] = np.log1p(df_prod['salary_from_adj'])

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(df_prod['salary_from_log'], bins = 40) 
plt.title('распределение зарплат', fontsize=14)
plt.xlabel('Заработная плата', fontsize=14)
plt.ylabel('Количество вакансий', fontsize=14)

Итоговый датасет df_prod содержит информацию с log и нормализованными значениями salary_from_log

In [ ]:
df_prod['salary_from_log'].skew()

In [ ]:

cat = ['role_name', 'schedule_id', 'employment_id','experience_ord']
target = 'salary_from_log'

df_plot = df_prod[cat + [target]].copy()

In [ ]:
#Boxplots для категориальных признаков
plt.figure(figsize=(24, 24))

for i, col in enumerate(cat, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x=col, y=target, data=df_plot)
    plt.xticks(rotation=45)
    plt.title(f'Salary by {col}')

plt.tight_layout()
plt.show()

Проведем первичный корреляционный анализ

In [ ]:
df_prod.shape

Т.к опыт - ранговая переменная, используем корреляцию Спирмена

In [ ]:
df_prod['experience_ord'].corr(df_prod['salary_from_log'], method='spearman')

In [ ]:
corr, p_value = spearmanr(df_prod['experience_ord'], df_prod['salary_from_log'])
print(corr, p_value)

Вывод Корреляционный анализ показал умеренную положительную статистически значимую связь между уровнем требуемого опыта и логарифмированной заработной платой (Spearman ρ = 0.388, p < 0.001). Полученный результат подтверждает экономическую гипотезу о росте заработной платы по мере увеличения профессионального опыта.

In [ ]:
groups = [
    group["salary_from_log"].values
    for name, group in df_prod.groupby("role_name")
]

stat, p = kruskal(*groups)
print(stat, p)

In [ ]:
groups = [
    group["salary_from_log"].values
    for name, group in df_prod.groupby("schedule_id")
]

stat, p = kruskal(*groups)
print(stat, p)

In [ ]:
groups = [
    group["salary_from_log"].values
    for name, group in df_prod.groupby("employment_id")
]

stat, p = kruskal(*groups)
print(stat, p)

**Вывод** Для оценки влияния номинальных категориальных признаков (профессиональная роль, тип графика работы, тип занятости) на уровень заработной платы использовался непараметрический критерий Краскела–Уоллиса.
Результаты показали статистически значимые различия между группами (p < 0.001), что свидетельствует о существенном влиянии данных факторов на формирование заработной платы.

Непараметрический критерий Краскела–Уоллиса выявил статистически значимые различия в уровне заработной платы между различными профессиональными ролями (H = 47269, p < 0.001), типами графика работы (H = 10295, p < 0.001) и типами занятости (H = 6134, p < 0.001).
Наибольший вклад в дифференциацию заработных плат вносит профессиональная роль, что соответствует экономической логике формирования оплаты труда.

In [ ]:
df_prod.shape

In [ ]:
df_prod.to_csv('external_data\df_prod.csv', index=False)

# Подготовка данных для M1

In [ ]:
features = ['role_name', 'experience_ord', 'schedule_id', 'employment_id','region_name']
target = 'salary_from_log'

df_m1_ols = df_prod[features + [target]].copy()

In [ ]:
#  M1 базовые проверки качества 
group_col = "region_name"
m1_snapshot = {
    "n_rows": int(df_m1_ols.shape[0]),
    "n_cols": int(df_m1_ols.shape[1]),
    "n_regions": int(df_m1_ols[group_col].nunique(dropna=True)),
    "target_name": target,
    "target_missing_pct": float(df_m1_ols[target].isna().mean() * 100),
    "rows_with_any_na_pct": float(df_m1_ols.isna().any(axis=1).mean() * 100),
}
display(pd.DataFrame([m1_snapshot]))

In [ ]:
df_m1_ols.duplicated().sum()

Найдено точных дублей: 91342

In [ ]:
#удалим точные дубликаты
df_m1_ols = df_m1_ols.drop_duplicates().reset_index(drop=True)
print(f"После удаления дубликатов: {df_m1_ols.shape[0]} строк")

In [ ]:
# Контрольный отчет после дедупликации M1:
post_dedup_report = pd.DataFrame({
    "metric": [
        "n_rows_post_dedup",
        "n_regions_post_dedup",
        "duplicates_remaining",
        "target_missing_pct_post_dedup"
    ],
    "value": [
        int(df_m1_ols.shape[0]),
        int(df_m1_ols["region_name"].nunique(dropna=True)),
        int(df_m1_ols.duplicated().sum()),
        float(df_m1_ols[target].isna().mean() * 100)
    ]
})
display(post_dedup_report)

Вывод: после удаления дублей регионы не потерялись

In [ ]:
group_sizes = df_m1_ols.groupby(features).size()
group_sizes.describe()

In [ ]:
# Проверка готовности M1-данных к GroupKFold по region_name
group_counts = (
    df_m1_ols.groupby("region_name", dropna=False)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
# Диагностика маленьких групп
display(group_counts.head(15))
min_group_size = int(group_counts["n_obs"].min())
n_groups = int(group_counts["region_name"].nunique(dropna=True))
# Рекомендуемое число фолдов: не больше минимального размера группы и не больше числа групп
recommended_n_splits = max(2, min(5, min_group_size, n_groups))
print(f"Min group size: {min_group_size}")
print(f"Number of groups: {n_groups}")
print(f"Recommended n_splits for GroupKFold: {recommended_n_splits}")
# Флаг потенциального риска для 5-fold
if min_group_size < 5:
    print("WARNING: Есть регионы с <5 наблюдениями. Для строгого GroupKFold(5) это методологический риск.")

In [ ]:
# Диагностика редких категорий в признаках M1

cat_cols = ["role_name", "schedule_id", "employment_id", "region_name"]
rare_threshold = 20  # можно поменять, например 10/20/30
rare_stats = []
for col in cat_cols:
    vc = df_m1_ols[col].value_counts(dropna=False)
    rare_cnt = int((vc < rare_threshold).sum())
    total_cnt = int(vc.shape[0])
    rare_share = float(rare_cnt / total_cnt * 100) if total_cnt > 0 else np.nan
    rare_stats.append({
        "feature": col,
        "n_categories": total_cnt,
        "n_categories_below_threshold": rare_cnt,
        "share_below_threshold_pct": round(rare_share, 2),
        "threshold": rare_threshold
    })
rare_stats_df = pd.DataFrame(rare_stats)
display(rare_stats_df)


**Итог подготовки данных для M1**:
* после дедупликации сформирован обучающий набор из 41 198 уникальных наблюдений;
* сохранены все 89 регионов, пропуски в признаках и таргете отсутствуют.
* Схема валидации GroupKFold(n_splits=5) по region_name применима
* (минимальный размер региональной группы = 6).

Диагностика редких категорий показывает низкий уровень разреженности:
* редкие уровни отсутствуют для role_name и schedule_id, ограниченно присутствуют
* для employment_id (1 из 5) и region_name (3 из 89), что является приемлемыдля M1 при использовании OneHotEncoder(handle_unknown='ignore').

In [ ]:
df_m1_ols.to_csv('data_for_models/df_m1_ols.csv', index=False)

# Подготовка данных для M2.1 Geo

In [ ]:
df_prod = pd.read_csv ('external_data/df_prod.csv', low_memory=False)

In [ ]:
df_prod_geo=df_prod.copy()

In [ ]:
# Получаем уникальные регионы
unique_regions = df_prod_geo['region_name'].dropna().unique()

# Сортируем для удобства
unique_regions = np.sort(unique_regions)
unique_regions = pd.DataFrame(unique_regions)

# Выводим количество и первые 20 регионов для проверки
print(f"Всего уникальных регионов: {len(unique_regions)}")
print(unique_regions[:20])


unique_regions.to_excel('unique_regions_2302.xlsx')

Загрузим справочник экономических макрорегионов и координат региональных центров

In [ ]:
reg_econ = pd.read_excel('external_data/reg_econ_0103v2.xlsx')

In [ ]:
reg_econ.head()

Посмотрим, есть ли пропуски в наименовании города и координат для 'Москва', 'Санкт-Петербург', 'Севастополь'

In [ ]:
regions_to_check = ['Москва', 'Санкт-Петербург', 'Севастополь']
sns.heatmap(df_prod_geo[df_prod_geo['region_name'].isin(regions_to_check)].isna().T)
plt.title('Тепловая карта пропусков значений')
plt.show()

In [ ]:
regions_to_check = ['Москва', 'Санкт-Петербург', 'Севастополь']

missing_city = df_prod_geo.loc[
    df_prod_geo['region_name'].isin(regions_to_check) & df_prod_geo['address.city_new'].isna(),
    ['region_name', 'address.city_new']
]

missing_lat = df_prod_geo.loc[
    df_prod_geo['region_name'].isin(regions_to_check) & df_prod_geo['address.lat_2'].isna(),
    ['region_name', 'address.lat_2']
]

missing_lon = df_prod_geo.loc[
    df_prod_geo['region_name'].isin(regions_to_check) & df_prod_geo['address.lng_2'].isna(),
    ['region_name', 'address.lng_2']
]

print("Пропуски в address.city_new:\n", missing_city)
print("Пропуски в address.lat_2:\n", missing_lat)
print("Пропуски в address.lng_2:\n", missing_lon)

In [ ]:
df_prod_geo['region_group'] = df_prod_geo['region_name'].apply(
    lambda x: x if x in regions_to_check else 'Все остальные регионы'
)
summary_missing = (
    df_prod_geo
    .groupby('region_group')[['address.city_new', 'address.lat_2', 'address.lng_2']]
    .apply(lambda x: x.isna().sum())
)

summary_missing

Пропуски есть только для Мосвы и Санкт-Петербурга, заполним их значениями, соответсвующими этим регионам

In [ ]:
regions_to_fill = ['Москва', 'Санкт-Петербург']

In [ ]:
#Подгрузим региональные центры и их координаты
df_prod_geo = df_prod_geo.merge(
    reg_econ[['region_name', 'economic_region','center_lat', 'center_lon']],
    on='region_name',
    how='left'
)

In [ ]:
df_prod_geo['economic_region'].isna().mean()

 заполним пропуски для 'Москва', 'Санкт-Петербург' по данным этих регионов

In [ ]:
mask = (
    df_prod_geo['region_name'].isin(regions_to_fill)
    & df_prod_geo['address.city_new'].isna()
)

df_prod_geo.loc[mask, 'address.city_new'] = df_prod_geo.loc[mask, 'region_name']

In [ ]:
mask_lat = (
    df_prod_geo['region_name'].isin(regions_to_fill)
    & df_prod_geo['address.lat_2'].isna()
)

df_prod_geo.loc[mask_lat, 'address.lat_2'] = df_prod_geo.loc[mask_lat, 'center_lat']

mask_lon = (
    df_prod_geo['region_name'].isin(regions_to_fill)
    & df_prod_geo['address.lng_2'].isna()
)

df_prod_geo.loc[mask_lon, 'address.lng_2'] = df_prod_geo.loc[mask_lon, 'center_lon']

summary_missing = (
    df_prod_geo
    .groupby('region_group')[['address.city_new', 'address.lat_2', 'address.lng_2']]
    .apply(lambda x: x.isna().sum())
)
summary_missing

Убедились, что сейчас для Москва, Санкт-Петербург и Севастополь	пропусков в address.city_new	address.lat_2	address.lng_2 нет

In [ ]:
df_prod_geo['lat_missing'] = df_prod_geo[['address.lat_2', 'address.lng_2']].isna().any(axis=1)
df_prod_geo['geo_available'] = (~df_prod_geo[['address.lat_2', 'address.lng_2']].isna().any(axis=1)).astype(int)

In [ ]:
print(df_prod_geo[['lat_missing', 'geo_available']].value_counts())

In [ ]:
#Переведем координаты в радианы
df_prod_geo['lat_rad'] = np.radians(df_prod_geo['address.lat_2'])
df_prod_geo['lon_rad'] = np.radians(df_prod_geo['address.lng_2'])

df_prod_geo['lat_sin'] = np.sin(df_prod_geo['lat_rad'])
df_prod_geo['lat_cos'] = np.cos(df_prod_geo['lat_rad'])

df_prod_geo['lon_sin'] = np.sin(df_prod_geo['lon_rad'])
df_prod_geo['lon_cos'] = np.cos(df_prod_geo['lon_rad'])

In [ ]:
df_prod_geo[['address.lat_2',
        'lat_sin', 'lat_cos',
        'address.lng_2',
        'lon_sin', 'lon_cos']].head()

In [ ]:
df_prod_geo[['lat_sin','lat_cos','lon_sin','lon_cos']].describe()

In [ ]:
#Создадим geohash для тех строк, где есть координаты
def safe_geohash(lat, lon, precision):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    return geohash2.encode(lat, lon, precision=precision)

In [ ]:
#Создадим несколько уровней geohash
df_prod_geo['geohash_4'] = df_prod_geo.apply(
    lambda row: safe_geohash(row['address.lat_2'], row['address.lng_2'], 4),
    axis=1
)

df_prod_geo['geohash_5'] = df_prod_geo.apply(
    lambda row: safe_geohash(row['address.lat_2'], row['address.lng_2'], 5),
    axis=1
)

df_prod_geo['geohash_6'] = df_prod_geo.apply(
    lambda row: safe_geohash(row['address.lat_2'], row['address.lng_2'], 6),
    axis=1
)

In [ ]:
df_prod_geo[['address.lat_2', 'address.lng_2', 
        'geohash_4', 'geohash_5', 'geohash_6']].head()

In [ ]:
df_prod_geo[['geohash_4','geohash_5','geohash_6']].nunique()

Определим расстояние до центра региона (координаты центров регионов мы подгрузили ранее)

In [ ]:
#проверим что пропусков нет
df_prod_geo[['center_lat', 'center_lon']].isna().mean()

In [ ]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371  # км
    
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

In [ ]:
#Расчёт distance_to_reg_center
#Считаем только там, где координаты есть
df_prod_geo['distance_to_reg_center'] = np.where(
    df_prod_geo['geo_available'] == 1,
    haversine_vectorized(
        df_prod_geo['address.lat_2'],
        df_prod_geo['address.lng_2'],
        df_prod_geo['center_lat'],
        df_prod_geo['center_lon']
    ),
    np.nan
)

In [ ]:
df_prod_geo['distance_to_reg_center'].describe()

In [ ]:
#Проверим
df_prod_geo.sort_values('distance_to_reg_center', ascending=True)[
    ['region_name', 'distance_to_reg_center']
].iloc[105000,:]

In [ ]:
#добавим логарифм расстояний
df_prod_geo['log_distance_to_reg_center'] = np.log1p(df_new2['distance_to_reg_center'])

In [ ]:
#проверим
df_prod_geo[df_prod_geo['region_name']=='Санкт-Петербург'][['region_name','address.lat_2','address.lng_2','center_lat','center_lon','distance_to_reg_center']]

In [ ]:
#для 'distance_to_reg_center' добавим бинарный индикатор отсутствия расстояния
df_prod_geo['distance_missing'] = df_prod_geo['distance_to_reg_center'].isna().astype(int)

In [ ]:
#заполним пропуски в address.city_new значением missing
df_prod_geo['address.city_new'] = df_prod_geo['address.city_new'].fillna('missing')


In [ ]:
# список геохеш-столбцов
geohash_cols = ['geohash_4', 'geohash_5', 'geohash_6']

#т.к catboost не умеет работать с пропусками в категориальных значениях, подставим вместо пропусков Missing
# заменяем все NaN на строку 'missing' и приводим к типу str
df_prod_geo[geohash_cols] = df_prod_geo[geohash_cols].fillna('missing').astype(str)


In [ ]:
#Проверим пропуски
plt.figure(figsize=(24,16))
sns.heatmap(df_prod_geo.isna().T)
plt.title('Тепловая карта пропусков значений')
plt.show()

In [ ]:
df_prod_geo.to_csv('external_data/df_prod_geo.csv', index=False)


## Morans

In [ ]:
# берем только строки где есть координаты
df_moran = df_prod_geo.dropna(subset=['address.lat_2','address.lng_2','salary_from_log']).copy()

In [ ]:
df_moran.shape

In [ ]:
df_moran.duplicated(subset=['address.lat_2', 'address.lng_2', 'salary_from_log']).sum()

In [ ]:
df_moran = df_prod_geo[['address.lat_2','address.lng_2','salary_from_log']].dropna()
df_moran = df_moran.drop_duplicates(subset=['address.lat_2', 'address.lng_2', 'salary_from_log'])

In [ ]:
df_moran.shape

In [ ]:
coords = df_moran[['address.lat_2','address.lng_2']].values
y = df_moran['salary_from_log'].values

#рассчитаем глобальный Moran
#10 ближайших соседей
w = KNN.from_array(coords, k=10)
w.transform = 'r'

mi = Moran(y, w)

print("Moran's I:", mi.I)
print("p-value:", mi.p_sim)

In [ ]:
# рассчитаем spatial lag для графика
# spatial lag = среднее значение соседей по весам
y_lag = np.zeros_like(y)
for i, neighbors in enumerate(w.neighbors.values()):
    y_lag[i] = np.mean(y[list(neighbors)])

# стандартизируем значения для визуализации
y_z = (y - np.mean(y)) / np.std(y)
y_lag_z = (y_lag - np.mean(y_lag)) / np.std(y_lag)

#построим Moran Scatterplot
plt.figure(figsize=(7,7))
plt.scatter(y_z, y_lag_z, alpha=0.5)
plt.axhline(0, color='grey', linestyle='--')
plt.axvline(0, color='grey', linestyle='--')
plt.xlabel('Standardized Salary (z-score)')
plt.ylabel('Spatial Lag (Average of neighbors, z-score)')
plt.title("Moran Scatterplot of Salary")
plt.grid(True)
plt.show()

In [ ]:
# рассчитаем Локальный Moran на том же датасете
df_local = df_moran.copy()
values = df_local['salary_from_log'].values

In [ ]:
df_local.info()

In [ ]:
# cоздаем весовую матрицу 10 ближайших соседей
w = KNN.from_array(coords, k=10)
w.transform = 'r'  # row-standardization

In [ ]:
# рассчитаем значения локального Moran's I с фиксированным seed для воспроизводимости
local_moran = Moran_Local(values, w, permutations=999, seed=7)

In [ ]:
# добавляем локальные значения Moran в датафрейм
df_local['local_moran_I'] = local_moran.Is
df_local['local_moran_p'] = local_moran.p_sim
df_local['local_moran_quadrant'] = local_moran.q

In [ ]:
# cоздаем булевы маски для каждого типа кластеров 
hotspots = (local_moran.q == 1) & (local_moran.p_sim < 0.05)  # HH
coldspots = (local_moran.q == 3) & (local_moran.p_sim < 0.05)  # LL
doughnuts = (local_moran.q == 2) & (local_moran.p_sim < 0.05)  # LH
diamonds = (local_moran.q == 4) & (local_moran.p_sim < 0.05)  # HL

In [ ]:
# распределение кластеров среди значимых точек
sig_df = df_local[df_local['local_moran_p'] < 0.05]
cluster_counts = sig_df['local_moran_quadrant'].value_counts()
print("Распределение локальных кластеров:\n", cluster_counts)

In [ ]:
center_lat = 61.52401
center_lon = 105.318756

m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="CartoDB positron")

for idx, row in sig_df.iterrows():
    q = row['local_moran_quadrant']
    
    # Показываем только HH и LL на карте
    if q not in [1,3]:
        continue
    
    lat = row['address.lat_2']
    lon = row['address.lng_2']
    
    if q == 1:
        color = "red"
        label = "High-High cluster (High salary area)"
    else:
        color = "blue"
        label = "Low-Low cluster (Low salary area)"

    popup_text = f"""
    Cluster: {label}<br>
    Salary log: {row['salary_from_log']:.2f}<br>
    Local Moran I: {row['local_moran_I']:.3f}
    """

    folium.CircleMarker(
        location=[lat, lon],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=popup_text
    ).add_to(m)
    
legend_html = '''
<div style="
position: fixed; 
bottom: 50px; left: 50px; width: 180px; height: 90px; 
border:2px solid grey; z-index:9999; font-size:14px;
background-color:white;
">
&nbsp;<b>LISA Cluster Legend</b><br>
&nbsp;<span style="background-color:red;color:red;">....</span>&nbsp;High-High (HH)<br>
&nbsp;<span style="background-color:blue;color:blue;">....</span>&nbsp;Low-Low (LL)
</div>
'''
m.get_root().html.add_child(Element(legend_html))
# m — объект карты с HH и LL кластерами
m

## Создадим итоговый датасет для M2.1

In [ ]:
feature_geo = [
    'experience_ord',
    'role_name',
    'schedule_id',
    'employment_id',
    'economic_region',
    'geo_available',
    'region_name',
    'address.city_new',
    'lat_sin',
    'lat_cos',
    'lon_sin',
    'lon_cos',
    'geohash_4',
    'geohash_5',
    'geohash_6',
    'distance_to_reg_center',
    'distance_missing']

target = 'salary_from_log'

df_m2_geo = df_prod_geo[feature_geo + [target]].copy()

### Контроль качества витрины M2.1 (df_m2_geo)

In [ ]:
#сделаем краткий QA-снимок df_m2_geo до дедупликации: размер выборки, регионы, пропуски по таргету и по строкам
group_col = "region_name"
m2_snapshot = {
    "n_rows": int(df_m2_geo.shape[0]),
    "n_cols": int(df_m2_geo.shape[1]),
    "n_regions": int(df_m2_geo[group_col].nunique(dropna=True)),
    "target_name": target,
    "target_missing_pct": float(df_m2_geo[target].isna().mean() * 100),
    "rows_with_any_na_pct": float(df_m2_geo.isna().any(axis=1).mean() * 100),
}
display(pd.DataFrame([m2_snapshot]))

In [ ]:
#удалим полные дубликаты строк и зафиксируем, сколько строк убрано
n_dup = int(df_m2_geo.duplicated().sum())
print(f"Точные дубликаты (полные строки): {n_dup}")
df_m2_geo = df_m2_geo.drop_duplicates().reset_index(drop=True)
print(f"После удаления дубликатов: {df_m2_geo.shape[0]} строк")

In [ ]:
#сделаем контроль после дедупа: финальный размер выборки, число регионов, остаток дублей, доля пропусков в таргете
post_dedup_report_m2 = pd.DataFrame({
    "metric": [
        "n_rows_post_dedup",
        "n_regions_post_dedup",
        "duplicates_remaining",
        "target_missing_pct_post_dedup",
    ],
    "value": [
        int(df_m2_geo.shape[0]),
        int(df_m2_geo["region_name"].nunique(dropna=True)),
        int(df_m2_geo.duplicated().sum()),
        float(df_m2_geo[target].isna().mean() * 100),
    ],
})
display(post_dedup_report_m2)

In [ ]:
#выполним  проверку размеров групп по region_name для GroupKFold: минимум наблюдений на регион и разумное число сплитов
group_counts_m2 = (
    df_m2_geo.groupby("region_name", dropna=False)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
display(group_counts_m2.head(15))
min_group_size = int(group_counts_m2["n_obs"].min())
n_groups = int(group_counts_m2["region_name"].nunique(dropna=False))
recommended_n_splits = max(2, min(5, min_group_size, n_groups))
print(f"Min group size: {min_group_size}")
print(f"Number of groups: {n_groups}")
print(f"Recommended n_splits for GroupKFold: {recommended_n_splits}")
if min_group_size < 5:
    print(
        "WARNING: Есть регионы c <5 наблюдениями. "
        "Для строгого GroupKFold(5) это методологический риск."
    )


In [ ]:
#выполним диагностику редких уровней в категориальных признаках M2 (порог по числу наблюдений на категорию)
cat_cols_m2 = [
    "role_name",
    "schedule_id",
    "employment_id",
    "region_name",
    "economic_region",
    "address.city_new",
    "geohash_4",
    "geohash_5",
    "geohash_6",
]
rare_threshold = 20  # как в M1
rare_stats_m2 = []
for col in cat_cols_m2:
    if col not in df_m2_geo.columns:
        continue
    vc = df_m2_geo[col].value_counts(dropna=False)
    rare_cnt = int((vc < rare_threshold).sum())
    total_cnt = int(vc.shape[0])
    rare_share = float(rare_cnt / total_cnt * 100) if total_cnt > 0 else np.nan
    rare_stats_m2.append({
        "feature": col,
        "n_categories": total_cnt,
        "n_categories_below_threshold": rare_cnt,
        "share_below_threshold_pct": round(rare_share, 2),
        "threshold": rare_threshold,
    })
display(pd.DataFrame(rare_stats_m2))

In [ ]:
df_m2_geo.info()

Итог подготовки данных для M2 geo:

* После дедупликации сформирован обучающий набор из 108 728 уникальных наблюдений (18 признаков и таргет);
* Сохранены все 89 регионов; пропусков в таргете нет; доля строк с хотя бы одним пропуском по признакам — ≈17,7 %;
* Схема валидации GroupKFold(n_splits=5) по region_name применима (минимальный размер региональной группы = 6).

Диагностика редких категорий (порог 20 наблюдений на уровень):
* для role_name, schedule_id и economic_region редких уровней нет;
* для employment_id и region_name редкие уровни есть, но это приемлимо для catboost;
* для address.city_new и geohash_4/5/6 много уникальных значений, у большинства из них мало вакансий, поэтому по порогу 20 наблюдений таких 'редких' уровней много, но для геоданных это нормальная ситуация

### Проверка пространственных признаков на мультиколлинеарность

In [ ]:
vif_cols = ["lat_sin", "lat_cos", "lon_sin", "lon_cos", "distance_to_reg_center"]
#geo_available и distance_missing это служебные бинарные индикаторы пропусков,
# их VIF не анализируется вместе с непрерывными геопризнаками.
X_vif = df_m2_geo.loc[df_m2_geo["geo_available"] == 1, vif_cols].copy()
X_vif = X_vif.replace([np.inf, -np.inf], np.nan).dropna()
vif_data = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values("VIF", ascending=False)
display(vif_data)

**Вывод**:
VIF оценивает *линейную* мультиколлинеарность в постановке OLS на подвыборке с валидной геолокацией (`geo_available = 1`). Индикаторы пропусков (`geo_available`, `distance_missing`) в расчёт не включались, чтобы избежать вырожденных столбцов после `dropna()`.

Повышенные значения VIF для `lat_sin` / `lat_cos` и части `lon_sin` / `lon_cos` ожидаемы: это функционально связанные преобразования координат (циклическое кодирование) и географическая автокорреляция признаков. При этом `distance_to_reg_center` имеее умеренный VIF (≈1.2), что указывает на отсутствие сильной линейной избыточности этих переменных относительно выбранного набора.

Для этапа M2 (`CatBoostRegressor`) высокий VIF у sincos **не является** сам по себе основанием удалять признаки или вводить масштабирование: это предупреждение прежде всего для линейных спецификаций; деревья используют иные разбиения пространства признаков.

In [ ]:
#выгрузим финальную витрину df_m2_geo
df_m2_geo.to_csv("data_for_models/df_m2_geo.csv", index=False)

# Подготовка данных для M2.2 Macro

In [ ]:
df_prod_geo = pd.read_csv ('external_data/df_prod_geo.csv', low_memory=False)
df_prod_geo.info()

In [ ]:
df_prod_geo_macro = df_prod_geo.copy()

## Подготовка данных о РК

In [ ]:
#Список регионов, в которых есть РК
rk_regions = [
    "Республика Алтай",
    "Алтайский край",
    "Амурская область",
    "Архангельская область",
    "Астраханская область",
    "Республика Башкортостан",
    "Республика Бурятия",
    "Вологодская область",
    "Забайкальский край",
    "Камчатский край",
    "Кировская область",
    "Республика Коми",
    "Костромская область",
    "Красноярский край",
    "Курганская область",
    "Магаданская область",
    "Мурманская область",
    "Ненецкий АО",
    "Новосибирская область",
    "Омская область",
    "Оренбургская область",
    "Пермский край",
    "Приморский край",
    "Ростовская область",
    "Саратовская область",
    "Сахалинская область",
    "Свердловская область",
    "Республика Татарстан",
    "Томская область",
    "Республика Тыва",
    "Тюменская область",
    "Удмуртская Республика",
    "Хабаровский край",
    "Республика Хакасия",
    "Ханты-Мансийский АО - Югра",
    "Челябинская область",
    "Чукотский АО",
    "Ямало-Ненецкий АО",
    "Республика Саха (Якутия)",
    "Республика Дагестан",
    "Иркутская область",
    "Кабардино-Балкарская республика",
    "Республика Калмыкия",
    "Республика Карелия",
    "Кемеровская область"
]


In [ ]:
# Признак reg_rk: True, если region_name содержит одно из значений из списка
df_prod_geo_macro["reg_rk"] = df_prod_geo_macro["region_name"].isin(rk_regions)

In [ ]:
# Заполним значения RK по регионам (для случаев когда для всего региона единые значения РК)
rk_map = {
    "Ненецкий АО": 1.5,
    "Новосибирская область": 1.2,
    "Оренбургская область": 1.15,
    "Приморский край": 1.2,
    "Республика Башкортостан": 1.15,
    "Республика Хакасия": 1.3,
    "Удмуртская Республика": 1.15,
    "Челябинская область": 1.15,
    "Чукотский АО": 2.0,
    "Магаданская область": 1.7,
}
# Создаем столбец RK и заполняем по точному совпадению region_name
df_prod_geo_macro["RK"] = df_prod_geo_macro["region_name"].map(rk_map)

In [ ]:
#Для тех регионов, где РК нет, проставим значение  1
df_prod_geo_macro.loc[df_prod_geo_macro["reg_rk"] == False, "RK"] = 1

In [ ]:
print(df_prod_geo_macro["RK"].value_counts(dropna=False))

In [ ]:
# Для дальнейшей предобработки, выгрузим в отдельный датафрейм гео данные по тем строкам, где РК неизвестен
mask_missing_rk = df_prod_geo_macro["RK"].isna()

cols_for_manual = [
    "address.lat_2",
    "address.lng_2",
    "address.city_new",
    "region_name",
    "reg_rk",
    "RK",
]
# 3) Копия в отдельный датафрейм
df_rk_missing = df_prod_geo_macro.loc[mask_missing_rk, cols_for_manual].copy()

In [ ]:
df_rk_missing = df_rk_missing.drop_duplicates().copy()

In [ ]:
df_rk_missing.shape

In [ ]:
df_rk_missing.to_excel("unload data\df_rk_missing.xlsx", index=False)

В большинстве регионов заданы разные значения РК для разных районов и населенных пунктов. Определим районы

In [ ]:
# Возьмем уникальные координаты, чтобы не дублировать запросы  Nominatim
coords = (
    df_rk_missing[["address.lat_2", "address.lng_2"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

In [ ]:
# Зададим Геокодер + ограничение частоты запросов
geolocator = Nominatim(user_agent="mag_project_rayon_lookup")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1.1, max_retries=2)
def get_district(lat, lon):
    try:
        loc = reverse((lat, lon), language="ru", exactly_one=True, addressdetails=True, zoom=12)
        if loc is None:
            return None
        addr = loc.raw.get("address", {})
        district = (
            addr.get("city_district")
            or addr.get("county")
            or addr.get("suburb")
            or addr.get("state_district")
        )
        return district
    except Exception:
        return None

In [ ]:
tqdm.pandas()

In [ ]:
#сделаем тестовый прогон 
coords_test = coords.head(100).copy()
coords_test["district"] = coords_test.progress_apply(
    lambda r: get_district(r["address.lat_2"], r["address.lng_2"]),
    axis=1
)
display(coords_test.head(20))

In [ ]:
# Определяем район для каждой уникальной пары координат
coords["district"] = coords.progress_apply(
    lambda r: get_district(r["address.lat_2"], r["address.lng_2"]),
    axis=1
)


In [ ]:
coords.head()

In [ ]:
# Сделаем merge обратно
df_rk_missing = df_rk_missing.merge(
    coords,
    on=["address.lat_2", "address.lng_2"],
    how="left"
)

In [ ]:
df_rk_missing.to_excel("unload data\df_rk_missing_plus_district.xlsx", index=False)

In [ ]:
df_rk_filled = pd.read_excel("external_data\df_rk_missing_plus_district_new2.xlsx")

In [ ]:
# Загружаем файл с уже заполненным rk/RK
df_rk_filled = pd.read_excel("external_data\df_rk_missing_plus_district_new2.xlsx")
# Ключевые поля
key_cols = ["address.lat_2", "address.lng_2", "address.city_new", "region_name"]
# Нормализация ключа
def prep_key(df):
    out = df.copy()
    out["address.lat_2_key"] = pd.to_numeric(out["address.lat_2"], errors="coerce").round(6)
    out["address.lng_2_key"] = pd.to_numeric(out["address.lng_2"], errors="coerce").round(6)
    out["address.city_new_key"] = out["address.city_new"].astype(str).str.strip().str.lower()
    out["region_name_key"] = out["region_name"].astype(str).str.strip().str.lower()
    return out
left = prep_key(df_prod_geo_macro)
right = prep_key(df_rk_filled)
merge_keys = ["address.lat_2_key", "address.lng_2_key", "address.city_new_key", "region_name_key"]
# оставим только ключ + заполненный RK из внешнего файла
right_map = (
    right[merge_keys + ["RK"]]
    .dropna(subset=["RK"])
    .drop_duplicates(subset=merge_keys, keep="last")
    .rename(columns={"RK": "RK_from_file"})
)
# заполняем только те RK, которые пустые в df_prod_geo_macro
merged = left.merge(right_map, on=merge_keys, how="left")
before_na = merged["RK"].isna().sum()
merged.loc[merged["RK"].isna(), "RK"] = merged.loc[merged["RK"].isna(), "RK_from_file"]
after_na = merged["RK"].isna().sum()
print("RK было пустых до:", before_na)
print("RK осталось пустых после:", after_na)
print("Заполнено:", before_na - after_na)

In [ ]:
# возвращаем обратно в df_prod_geo_macro
drop_tmp = merge_keys + ["RK_from_file"]
df_prod_geo_macro = merged.drop(columns=drop_tmp)

In [ ]:
print(df_prod_geo_macro["RK"].value_counts(dropna=False))

In [ ]:
df_prod_geo_macro.info()

In [ ]:
#Проверим пропуски
plt.figure(figsize=(24, 16))
sns.heatmap(df_prod_geo_macro.isna().T)
plt.title('Тепловая карта пропусков значений')
plt.show()

Для всех строк заполнены данные с РК. Проанализируем распределение РК

In [ ]:
# Проверм распределение РК по уровням
COL_RK = "RK"
Y_COL = "salary_from_log"
assert COL_RK in df_prod_geo_macro.columns
assert Y_COL in df_prod_geo_macro.columns
rk = pd.to_numeric(df_prod_geo_macro[COL_RK])
rk_clean = rk.dropna()
levels_sorted = np.sort(rk_clean.unique())
K = int(len(levels_sorted))
freq = (
    rk_clean.value_counts()
    .reindex(levels_sorted)
    .rename("n")
    .to_frame()
)
freq["share"] = freq["n"] / freq["n"].sum()
freq["cum_share"] = freq["share"].cumsum()
display(freq.round(6))
if 1.0 in freq.index:
    print(f"Доля RK == 1.00: {float(freq.loc[1.0, 'share']):.4%}")
for thr in (30, 50, 100):
    n_rare = int((freq["n"] < thr).sum())
    print(f"Уровней  n < {thr}: {n_rare} / {K}")
tail_thr = 1.60
mask_tail = freq.index.astype(float) >= tail_thr
print(f"Доля наблюдений  RK >= {tail_thr}: {float(freq.loc[mask_tail, 'share'].sum()):.4%}")

In [ ]:
# Проверим связь с salary_from_log
d = df_prod_geo_macro[[COL_RK, Y_COL]].copy()
d[COL_RK] = pd.to_numeric(d[COL_RK], errors="coerce")
d = d.dropna(subset=[COL_RK, Y_COL])
rho, p_sp = stats.spearmanr(d[COL_RK], d[Y_COL])
groups = [g[Y_COL].values for _, g in d.groupby(COL_RK, sort=True)]
all_y = d[Y_COL].values
grand_mean = all_y.mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = ((all_y - grand_mean) ** 2).sum()
eta2 = ss_between / ss_total if ss_total > 0 else np.nan
print(f"Spearman rho = {rho:.4f}, p = {p_sp:.3e}")
print(f"η² (между уровнями RK) = {eta2:.6f}")
by_rk = (
    d.groupby(COL_RK, observed=True)[Y_COL]
    .agg(n="count", mean="mean", median="median", std="std")
    .sort_index()
)
by_rk["se"] = by_rk["std"] / np.sqrt(by_rk["n"])
display(by_rk.round(6))

In [ ]:
# Построим графики
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(freq))
axes[0].bar(x, freq["n"].values, color="#2c5378", edgecolor="white", linewidth=0.4)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"{v:.2f}" for v in freq.index.astype(float)], rotation=45, ha="right")
axes[0].set_title("Частоты по уровням РК")
axes[0].set_ylabel("n")
axes[1].plot(x, freq["cum_share"].values, drawstyle="steps-mid", color="#8b2f39", linewidth=2)
axes[1].scatter(x, freq["cum_share"].values, color="#8b2f39", s=22, zorder=3)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{v:.2f}" for v in freq.index.astype(float)], rotation=45, ha="right")
axes[1].set_ylim(0, 1.01)
axes[1].set_title("Накопленная доля по возрастанию РК")
axes[1].set_ylabel("cum_share")
plt.tight_layout()
plt.show()
fig2, ax = plt.subplots(figsize=(9, 4.5))
x_rk = by_rk.index.astype(float).values
y_mean = by_rk["mean"].values
y_se = by_rk["se"].values
ax.errorbar(
    x_rk,
    y_mean,
    yerr=y_se,
    fmt="o",
    color="#1f6f54",
    ecolor="gray",
    capsize=4,
    markersize=8,
    label="Среднее ± SE",
)
# Линейный тренд по агрегированным точкам
if len(x_rk) >= 2:
    coef = np.polyfit(x_rk, y_mean, 1)
    x_line = np.linspace(x_rk.min(), x_rk.max(), 100)
    y_line = coef[0] * x_line + coef[1]
    ax.plot(
        x_line,
        y_line,
        "--",
        color="#8b2f39",
        linewidth=2,
        alpha=0.9,
        label=f"Линейный тренд (OLS по точкам): наклон {coef[0]:.4f}",
    )
ax.set_xlabel("RK")
ax.set_ylabel(Y_COL)
ax.set_title(f"Среднее {Y_COL} по уровням РК")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

**Вывод**:
* РК в выборке принимает несколько фиксированных уровней. Распределение сильно скошено к RK = 1,00 (~69% наблюдений), при этом ~95% данных не выше 1,20. Доля RK ≥ 1,60 очень мала (~0,21%)
* Связь с salary_from_log: Связь уровня РК с salary_from_log на всех наблюдениях слабая: по рангам она почти не видна, средние по группам уровней РК меняются неровно. В графике показан линейный тренд по средним на каждом уровне RK, но без учета структурных признаков (например, профессиоональной роли), вывод о влиянии РК на зарплату пока сделать нельзя

* В модели M2.5 РК будем включать как числовой признак, дополнительное бинирование и ранжирование использовать не будем, т.к РК уже задан дискретными уровнями
* Шкалировать РК не будем, т.к. CatBoost (M2.5) строит разбиения по порогам исходных признаков, поэтому стандартизация РК не нужна.

## Подготовка данных о валовом региональном продукте

In [ ]:
grp_u_2023 = pd.read_excel ('external_data/GRP_U_2023.xlsx')

In [ ]:
grp_u_2023.info()

In [ ]:
grp_u_2023.head()

In [ ]:
#Проанализируем, для каких регионов не заданы значения валового продукта и безработицы
regions_in_main = set(df_prod_geo_macro["region_name"].dropna().unique())
regions_in_macro = set(grp_u_2023["region_name"].dropna().unique())
regions_no_macro = sorted(regions_in_main - regions_in_macro)
print(regions_no_macro)

In [ ]:
for col in ("GRP_K", "U_delta"):
    if col in df_prod_geo_macro.columns:
        df_prod_geo_macro = df_prod_geo_macro.drop(columns=[col])
macro_for_merge = grp_u_2023[["region_name", "GRP_K", "U_delta"]].copy()
df_prod_geo_macro = df_prod_geo_macro.merge(macro_for_merge, on="region_name", how="left")
macro_missing = df_prod_geo_macro["GRP_K"].isna()
print("Доля наблюдений без валового продукта и безработицы до импутации:", f"{macro_missing.mean():.4%}")

Для регионов, где не заданы значения валового продукта и безработицы импутируем значения на уровне РФ

In [ ]:
U_RF = 0.6 # Уровень зарегистрированной безработицы РФ 2023
GRP_RF = 1_073_650.9 #Валовой Ррегиональный продукт на душу населения РФ 2023г

In [ ]:
fill_grp_k = 1.0
fill_u_delta = 0.0
df_prod_geo_macro["macro_k_u_imputed"] = macro_missing.astype(np.int8)
df_prod_geo_macro["GRP_K"] = df_prod_geo_macro["GRP_K"].fillna(fill_grp_k)
df_prod_geo_macro["U_delta"] = df_prod_geo_macro["U_delta"].fillna(fill_u_delta)
assert df_prod_geo_macro[["GRP_K", "U_delta"]].isna().sum().sum() == 0
print(
    f"Импутация: GRP_K={fill_grp_k}, U_delta={fill_u_delta} "
    f"(справочно U_RF={U_RF}%, GRP_RF={GRP_RF})"
)
print("Наблюдений с импутированными значениями валового продукта и безработицы:", int(df_prod_geo_macro["macro_k_u_imputed"].sum()))

In [ ]:
Y_COL = "salary_from_log"  # как в блоке РК
assert Y_COL in df_prod_geo_macro.columns
flag = "macro_k_u_imputed"
cols = ["GRP_K", "U_delta", Y_COL]
print("Доля импутированных макро:", df_prod_geo_macro[flag].mean())
print("\ndescribe() — все строки:")
display(df_prod_geo_macro[cols].describe())

In [ ]:
for part, m in [("все", None), ("без импутации", df_prod_geo_macro[flag] == 0)]:
    sub = df_prod_geo_macro if m is None else df_prod_geo_macro.loc[m]
    print(f"\nSpearman  {Y_COL} ({part}):")
    print(sub[["GRP_K", "U_delta"]].corrwith(sub[Y_COL], method="spearman"))

In [ ]:
# Гистограммы
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df_prod_geo_macro["GRP_K"], kde=True, ax=ax[0])
ax[0].set_title("GRP_K")
sns.histplot(df_prod_geo_macro["U_delta"], kde=True, ax=ax[1])
ax[1].set_title("U_delta")
plt.tight_layout()
plt.show()

In [ ]:
# Средние по квантильным группам GRP_K (аналог «уровней РК»)
q =10
df_prod_geo_macro["_grp_k_bin"] = pd.qcut(df_prod_geo_macro["GRP_K"], q, duplicates="drop")
mgrp = df_prod_geo_macro.groupby("_grp_k_bin", observed=True)[Y_COL].agg(["mean", "median", "count"])
display(mgrp)
df_prod_geo_macro.drop(columns="_grp_k_bin", inplace=True)

In [ ]:
# То же для U_delta 
r = 8
df_prod_geo_macro["_ud_bin"] = pd.qcut(df_prod_geo_macro["U_delta"], r, duplicates="drop")
mud = df_prod_geo_macro.groupby("_ud_bin", observed=True)[Y_COL].agg(["mean", "median", "count"])
display(mud)
df_prod_geo_macro.drop(columns="_ud_bin", inplace=True)

**Вывод**
* Spearman: GRP_K ~0,39 с salary_from_log связь умеренная; U_delta ~−0,17 связь слабая отрицательная. Это сильнее, чем 'почти ноль' по РК
* Для GRP_K и U_delta в модели M2.5 используем исходные значения, без бинаризации (CatBoost сам подбирает пороги разбиений), а жёсткая дискретизация по квантилям этой выборки теряет информацию и не даёт устойчивых границ между группами.
* Шкалировать GRP_K и U_delta не будем по той же логике, что и РК: для CatBoost стандартизация не требуется
* Для модели CatBoost (M2.5) так же будем использовать признак macro_k_u_imputed, для индикации строк, в которые сделали импутацию GRP_K = 1 и U_delta = 0 по значениям РФ



In [ ]:
df_prod_geo_macro.info()

In [ ]:
df_prod_geo_macro.to_csv('external_data/df_prod_geo_macro.csv', index=False)

## Создадим итоговый датасет для M2.2

In [ ]:
feature_macro = [
    'experience_ord',
    'role_name',
    'schedule_id',
    'employment_id',
    'region_name',
    'RK',
    'GRP_K',
    'U_delta',
    'macro_k_u_imputed']

target = 'salary_from_log'

df_m2_macro = df_prod_geo_macro[feature_macro + [target]].copy()

### Контроль качества витрины M2.2 (df_m2_macro)

In [ ]:
#сделаем краткий QA-снимок df_m2_macro до дедупликации: размер выборки, регионы, пропуски по таргету и по строкам
group_col = "region_name"
m2_snapshot = {
    "n_rows": int(df_m2_macro.shape[0]),
    "n_cols": int(df_m2_macro.shape[1]),
    "n_regions": int(df_m2_macro[group_col].nunique(dropna=True)),
    "target_name": target,
    "target_missing_pct": float(df_m2_macro[target].isna().mean() * 100),
    "rows_with_any_na_pct": float(df_m2_macro.isna().any(axis=1).mean() * 100),
}
display(pd.DataFrame([m2_snapshot]))

In [ ]:
#удалим полные дубликаты строк и зафиксируем, сколько строк убрано
n_dup = int(df_m2_macro.duplicated().sum())
print(f"Точные дубликаты (полные строки): {n_dup}")
df_m2_macro = df_m2_macro.drop_duplicates().reset_index(drop=True)
print(f"После удаления дубликатов: {df_m2_macro.shape[0]} строк")

In [ ]:
#сделаем контроль после дедупа: финальный размер выборки, число регионов, остаток дублей, доля пропусков в таргете
post_dedup_report_m2 = pd.DataFrame({
    "metric": [
        "n_rows_post_dedup",
        "n_regions_post_dedup",
        "duplicates_remaining",
        "target_missing_pct_post_dedup",
    ],
    "value": [
        int(df_m2_macro.shape[0]),
        int(df_m2_macro["region_name"].nunique(dropna=True)),
        int(df_m2_macro.duplicated().sum()),
        float(df_m2_macro[target].isna().mean() * 100),
    ],
})
display(post_dedup_report_m2)

In [ ]:
#выполним  проверку размеров групп по region_name для GroupKFold: минимум наблюдений на регион и разумное число сплитов
group_counts_m2 = (
    df_m2_macro.groupby("region_name", dropna=False)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
display(group_counts_m2.head(15))
min_group_size = int(group_counts_m2["n_obs"].min())
n_groups = int(group_counts_m2["region_name"].nunique(dropna=False))
recommended_n_splits = max(2, min(5, min_group_size, n_groups))
print(f"Min group size: {min_group_size}")
print(f"Number of groups: {n_groups}")
print(f"Recommended n_splits for GroupKFold: {recommended_n_splits}")
if min_group_size < 5:
    print(
        "WARNING: Есть регионы c <5 наблюдениями. "
        "Для строгого GroupKFold(5) это методологический риск."
    )


In [ ]:
#выполним диагностику редких уровней в категориальных признаках M2 (порог по числу наблюдений на категорию)
cat_cols_m2 = [
    "role_name",
    "schedule_id",
    "employment_id",
    "region_name",
    "economic_region",
    "address.city_new",
    "geohash_4",
    "geohash_5",
    "geohash_6",
]
rare_threshold = 20  # как в M1
rare_stats_m2 = []
for col in cat_cols_m2:
    if col not in df_m2_macro.columns:
        continue
    vc = df_m2_macro[col].value_counts(dropna=False)
    rare_cnt = int((vc < rare_threshold).sum())
    total_cnt = int(vc.shape[0])
    rare_share = float(rare_cnt / total_cnt * 100) if total_cnt > 0 else np.nan
    rare_stats_m2.append({
        "feature": col,
        "n_categories": total_cnt,
        "n_categories_below_threshold": rare_cnt,
        "share_below_threshold_pct": round(rare_share, 2),
        "threshold": rare_threshold,
    })
display(pd.DataFrame(rare_stats_m2))

In [ ]:
df_m2_macro.info()

In [ ]:
#Проверим признаки на мультиколлинеарность
num_cols = ["experience_ord", "RK", "GRP_K", "U_delta", "macro_k_u_imputed"]
corr_p = df_m2_macro[num_cols].corr(method="pearson")
corr_s = df_m2_macro[num_cols].corr(method="spearman")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(corr_p, annot=True, fmt=".3f", ax=ax[0], vmin=-1, vmax=1, center=0)
ax[0].set_title("Pearson")
sns.heatmap(corr_s, annot=True, fmt=".3f", ax=ax[1], vmin=-1, vmax=1, center=0)
ax[1].set_title("Spearman")
plt.tight_layout()
plt.show()
print("Pearson:\n", corr_p)
print("\nSpearman:\n", corr_s)

* По Пирсону между числовыми признаками сильной линейной мультиколлинеарности нет: самые заметные пары - RK с GRP_K и RK с U_delta (|r| ≈ 0,28).
* По Спирмену связь RK с U_delta сильнее (ρ ≈ 0,61), то есть по рангам региональные сигналы пересекаются сильнее, чем видно по линейной корреляции; это не запрещает использование обоих признаков, но при интерпретации стоит помнить.
* RK, GRP_K и U_delta оставляем в модели: у них разный экономический смысл; при разборе важности признаков и SHAP в CatBoost будем учитывать, что часть регионального эффекта они могут нести совместно.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df_m2_macro[num_cols].astype(float)
# при константном столбце VIF может разъехаться — при необходимости временно уберите macro_k_u_imputed
vif = pd.DataFrame({
    "feature": num_cols,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif.sort_values("VIF", ascending=False)

По VIF линейная мультиколлинеарность между числовыми признаками умеренная, все VIF < 5; максимум у RK (~3,1), что согласуется с корреляциями РК с макропризнаками. U_delta и macro_k_u_imputed в линейной постановке почти не избыточны.

In [ ]:
#выгрузим финальную витрину df_m2_macro
df_m2_macro.to_csv("data_for_models/df_m2_macro.csv", index=False)

# Подготовка данных для M2.3 Geo + Macro

In [ ]:
feature_geo_macro = [
     'experience_ord',
     'role_name',
     'schedule_id',
     'employment_id',
     'economic_region',
     'geo_available',
     'region_name',
     'address.city_new',
     'lat_sin',
     'lat_cos',
     'lon_sin',
     'lon_cos',
     'geohash_4',
     'geohash_5',
     'geohash_6',
     'distance_to_reg_center',
     'distance_missing',
     'RK',
     'GRP_K',
     'U_delta',
     'macro_k_u_imputed']

target = 'salary_from_log'

df_m2_geo_macro = df_prod_geo_macro[feature_geo_macro + [target]].copy()

In [ ]:
df_m2_geo_macro.info()

### Контроль качества витрины M2.3 (df_m2_geo_macro)

In [ ]:
#сделаем краткий QA-снимок df_m2_macro до дедупликации: размер выборки, регионы, пропуски по таргету и по строкам
group_col = "region_name"
m2_geo_macro_snapshot = {
    "n_rows": int(df_m2_geo_macro.shape[0]),
    "n_cols": int(df_m2_geo_macro.shape[1]),
    "n_regions": int(df_m2_geo_macro[group_col].nunique(dropna=True)),
    "target_name": target,
    "target_missing_pct": float(df_m2_geo_macro[target].isna().mean() * 100),
    "rows_with_any_na_pct": float(df_m2_geo_macro.isna().any(axis=1).mean() * 100),
}
display(pd.DataFrame([m2_geo_macro_snapshot]))

In [ ]:
#удалим полные дубликаты строк и зафиксируем, сколько строк убрано
n_dup = int(df_m2_geo_macro.duplicated().sum())
print(f"Точные дубликаты (полные строки): {n_dup}")
df_m2_geo_macro = df_m2_geo_macro.drop_duplicates().reset_index(drop=True)
print(f"После удаления дубликатов: {df_m2_geo_macro.shape[0]} строк")

In [ ]:
#сделаем контроль после дедупа: финальный размер выборки, число регионов, остаток дублей, доля пропусков в таргете
post_dedup_report = pd.DataFrame({
    "metric": [
        "n_rows_post_dedup",
        "n_regions_post_dedup",
        "duplicates_remaining",
        "target_missing_pct_post_dedup",
    ],
    "value": [
        int(df_m2_geo_macro.shape[0]),
        int(df_m2_geo_macro["region_name"].nunique(dropna=True)),
        int(df_m2_geo_macro.duplicated().sum()),
        float(df_m2_geo_macro[target].isna().mean() * 100),
    ],
})
display(post_dedup_report)

In [ ]:
#выполним  проверку размеров групп по region_name для GroupKFold: минимум наблюдений на регион и разумное число сплитов
group_counts = (
    df_m2_geo_macro.groupby("region_name", dropna=False)
    .size()
    .reset_index(name="n_obs")
    .sort_values("n_obs", ascending=True)
)
display(group_counts.head(15))
min_group_size = int(group_counts["n_obs"].min())
n_groups = int(group_counts["region_name"].nunique(dropna=False))
recommended_n_splits = max(2, min(5, min_group_size, n_groups))
print(f"Min group size: {min_group_size}")
print(f"Number of groups: {n_groups}")
print(f"Recommended n_splits for GroupKFold: {recommended_n_splits}")
if min_group_size < 5:
    print(
        "WARNING: Есть регионы c <5 наблюдениями. "
        "Для строгого GroupKFold(5) это методологический риск."
    )

In [ ]:
# Построим матрицы корреляций (Pearson / Spearman) по числовому подмножеству df_m2_geo_macro для M2.10.
num_cols = [
    "experience_ord",
    "geo_available",
    "lat_sin",
    "lat_cos",
    "lon_sin",
    "lon_cos",
    "distance_to_reg_center",
    "distance_missing",
    "RK",
    "GRP_K",
    "U_delta",
    "macro_k_u_imputed",
]
corr_p = df_m2_geo_macro[num_cols].corr(method="pearson")
corr_s = df_m2_geo_macro[num_cols].corr(method="spearman")
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(corr_p, annot=True, fmt=".2f", ax=ax[0], vmin=-1, vmax=1, center=0)
ax[0].set_title("Pearson")
sns.heatmap(corr_s, annot=True, fmt=".2f", ax=ax[1], vmin=-1, vmax=1, center=0)
ax[1].set_title("Spearman")
plt.tight_layout()
plt.show()
print("Pearson:\n", corr_p)
print("\nSpearman:\n", corr_s)

* По Пирсону сильной линейной связи между блоком «опыт + макро» (experience_ord, GRP_K, U_delta, macro_k_u_imputed) нет; для RK с GRP_K и RK с U_delta линейные коэффициенты умеренные (порядка |r| ≈ 0,12–0,37 в зависимости от пары), что согласуется с отдельным VIF по этому поднабору (все < 5).
* По Спирмену RK с U_delta заметно сильнее (ρ ≈ 0,65), чем по Пирсону (≈ 0,37): ранговая согласованность региональных сигналов выражена сильнее линейной; при интерпретации важности признаков и SHAP стоит помнить о совместном региональном вкладе.
* RK тесно согласован с lon_sin / lon_cos (по Пирсону ≈ 0,67–0,72, по Спирмену ≈ 0,69–0,73): категориальный региональный уровень и непрерывное положение частично несут один смысл — это ожидаемо и не требует выбрасывать один из каналов для CatBoost, но усложняет линейную интерпретацию «чистого» эффекта каждого признака.
* lat_sin и lat_cos дают |r| ≈ 0,99 (по Спирмену ранговая структура −1 / +1 на доступных наблюдениях) — типичная избыточность sin/cos для широты; согласуется с очень высоким VIF только по геочисловому блоку на geo_available == 1.
* Ячейки NaN в матрице у пар «geo_available — координаты / distance_to_reg_center» связаны с пропусками гео и pairwise расчётом корреляций, а не с отсутствием экономического смысла; geo_available и distance_missing дают жёсткую отрицательную линейную связь (r = −1), что отражает конструкцию индикаторов, а не новый факт для модели.

Итог: числовой набор M2.10 оставляем в модели; корреляции и NaN трактуем как описательную линейную диагностику и опору для текста, без автоматического отбора признаков под CatBoost по порогам |r| или по «совместному» VIF на гео-подвыборке.

In [ ]:

# оценим vif только пространственного блока (sin/cos координат и расстояния до центра региона)
vif_geo_cols = ["lat_sin", "lat_cos", "lon_sin", "lon_cos", "distance_to_reg_center"]
X_vif_geo = df_m2_geo_macro.loc[df_m2_geo_macro["geo_available"] == 1, vif_geo_cols].copy()
X_vif_geo = X_vif_geo.replace([np.inf, -np.inf], np.nan).dropna()
vif_geo = pd.DataFrame({
    "feature": X_vif_geo.columns,
    "VIF": [variance_inflation_factor(X_vif_geo.values, i) for i in range(X_vif_geo.shape[1])],
}).sort_values("VIF", ascending=False)
display(vif_geo)

VIF у sin/cos широты и долготы очень высокий (максимум у lat_sin и lon_sin), у distance_to_reg_center - умеренный (~1,2). Это ожидаемо для циклического кодирования координат и не является основанием убирать эти признаки в CatBoost

In [ ]:
# оценим vif по блоку структура + макро (experience_ord, RK, GRP_K, U_delta, macro_k_u_imputed) на всей выборке df_m2_geo_macro
vif_macro_num = ["experience_ord", "RK", "GRP_K", "U_delta", "macro_k_u_imputed"]
X_macro = df_m2_geo_macro[vif_macro_num].astype(float)
vif_macro = pd.DataFrame({
    "feature": vif_macro_num,
    "VIF": [variance_inflation_factor(X_macro.values, i) for i in range(X_macro.shape[1])],
}).sort_values("VIF", ascending=False)
display(vif_macro)

линейная мультиколлинеарность умеренная, все VIF < 5; максимумы у RK (~3,5) и GRP_K (~3,0), U_delta и macro_k_u_imputed низкие

In [ ]:
# оценим vif по объединённому числовому блоку (геочисловые + опыт + макро) на подвыборке geo_available == 1
vif_geo_macro_joint_cols = [
    "lat_sin", "lat_cos", "lon_sin", "lon_cos", "distance_to_reg_center",
    "experience_ord", "RK", "GRP_K", "U_delta", "macro_k_u_imputed",
]
X_vif_joint = df_m2_geo_macro.loc[df_m2_geo_macro["geo_available"] == 1, vif_geo_macro_joint_cols].copy()
X_vif_joint = X_vif_joint.replace([np.inf, -np.inf], np.nan).dropna()
print(f"Строк для VIF (geo_available==1, без NA): {len(X_vif_joint)}")
vif_geo_macro_joint = pd.DataFrame({
    "feature": X_vif_joint.columns,
    "VIF": [variance_inflation_factor(X_vif_joint.values, i) for i in range(X_vif_joint.shape[1])],
}).sort_values("VIF", ascending=False)
display(vif_geo_macro_joint)

VIF взрывается (в первую очередь у RK и lat_sin, порядка 10²). Это отражает совместную линейную избыточность «регион в виде РК + координаты + макро» на одной подвыборке

Для M2.10 CatBoost признаки оставляем; VIF используем для прозрачности и обсуждения регионально-пространственного перекрытия, а не как правило автоматического исключения признаков.

In [ ]:
df_m2_geo_macro.to_csv("data_for_models/df_m2_geo_macro.csv", index=False)